# Task 1 — Messy Dataset, Baseline Model

**MLSA SRM Technical Recruitment — AI/ML | Second Year**

A reasoning-first workflow: understand the messy data, fix only defensible problems, compare two simple models, and inspect errors. The goal is a model I can genuinely explain rather than an overengineered solution.

## 1. Problem statement

Predict whether an applicant completed the recruitment task (`completed_task`). The starter dataset intentionally contains missing values, inconsistent categorical representations, impossible numeric values, duplicates, and unusual numeric observations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
pd.set_option('display.max_columns', None)

## 2. Load and audit the raw data

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/MLSA-SRM/recruit-task-messy-dataset/main/recruitment_engagement.csv'
df = pd.read_csv(DATA_URL)
print('Shape:', df.shape)
display(df.head())
print('\nMissing values:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate applicant IDs:', df['applicant_id'].duplicated().sum())
display(df[['prep_hours_last_week','quiz_score','days_since_signup']].describe().T)

## 3. What is messy, and what will I fix?

**Categorical inconsistencies:** normalize equivalent labels such as `Technical`/`technical`/`TECHNICAL`, `First`/`1`/`1st Year`, and `Yes`/`yes`/`Y`.

**Impossible values:** negative preparation hours and negative days since signup are logically impossible, so they are converted to missing. Quiz scores outside 0–100 are also treated as missing.

**Missing values:** do not drop applicants just because a field is missing. Numeric values will be median-imputed and categorical values most-frequent-imputed inside the model pipeline.

**Outliers:** use IQR to flag unusual values, but do not automatically delete them. A very high preparation time may be unusual without being impossible.

**Identifiers:** `applicant_id` and `name` are excluded from model features because they identify people rather than describe engagement.

In [ ]:
clean = df.drop_duplicates().copy()
clean['domain'] = clean['domain'].astype('string').str.strip().str.lower().str.title()
clean['subdomain'] = clean['subdomain'].astype('string').str.strip().str.lower()
clean['subdomain'] = clean['subdomain'].replace({'ai/ml':'AI/ML','web dev':'Web Dev'}).str.title()
clean['subdomain'] = clean['subdomain'].replace({'Ai/Ml':'AI/ML','Pr':'PR'})
clean['year'] = clean['year'].astype('string').str.strip().str.lower().map({'first':'1st Year','1':'1st Year','1st year':'1st Year','second':'2nd Year','2':'2nd Year','2nd year':'2nd Year'})
clean['signup_source'] = clean['signup_source'].astype('string').str.strip().str.title()
clean['prior_experience'] = clean['prior_experience'].astype('string').str.strip().str.lower().map({'yes':'Yes','y':'Yes','no':'No','n':'No'})
for col in ['prep_hours_last_week','quiz_score','days_since_signup']:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')
clean.loc[clean['prep_hours_last_week'] < 0, 'prep_hours_last_week'] = np.nan
clean.loc[clean['days_since_signup'] < 0, 'days_since_signup'] = np.nan
clean.loc[~clean['quiz_score'].between(0,100), 'quiz_score'] = np.nan
clean['completed_task'] = clean['completed_task'].astype('string').str.strip().str.title()
print('Rows before:', len(df))
print('Rows after exact-duplicate removal:', len(clean))
display(clean.head())

## 4. Outlier check and quick EDA

IQR is used as a diagnostic. An outlier is not automatically an error.

In [ ]:
def iqr_outliers(series):
    s = series.dropna(); q1,q3 = s.quantile([0.25,0.75]); iqr=q3-q1
    lo,hi=q1-1.5*iqr,q3+1.5*iqr
    return lo,hi,s[(s<lo)|(s>hi)]
for col in ['prep_hours_last_week','quiz_score','days_since_signup']:
    lo,hi,out=iqr_outliers(clean[col]); print(f'{col}: IQR bounds=({lo:.2f}, {hi:.2f}), flagged={len(out)}')
print('\nTarget distribution:')
display(clean['completed_task'].value_counts(normalize=True).mul(100).round(1).to_frame('percent'))
display(clean.groupby('completed_task')[['prep_hours_last_week','quiz_score','days_since_signup']].median().round(2))
fig,ax=plt.subplots(figsize=(6,4)); clean.boxplot(column='quiz_score',by='completed_task',ax=ax); ax.set_title('Quiz score by task completion'); ax.set_xlabel('Completed task'); ax.set_ylabel('Quiz score'); plt.suptitle(''); plt.show()

## 5. Train/test split and preprocessing

The split happens before fitting imputers, scalers, and encoders so information from the test set cannot leak into training.

In [ ]:
X=clean.drop(columns=['completed_task','applicant_id','name']); y=clean['completed_task']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
numeric=X.select_dtypes(include=['number']).columns.tolist()
categorical=X.select_dtypes(include=['object','string']).columns.tolist()
preprocessor=ColumnTransformer([
    ('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',RobustScaler())]),numeric),
    ('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),categorical)
])
print('Numeric:',numeric)
print('Categorical:',categorical)
print('Train:',X_train.shape,'Test:',X_test.shape)

## 6. Compare two simple models

**Logistic Regression** is the interpretable linear baseline. **Random Forest** is a modest nonlinear baseline that can capture interactions. I am deliberately avoiding aggressive hyperparameter tuning because the dataset is small and the goal is a defensible baseline.

In [ ]:
models={
    'Logistic Regression':Pipeline([('preprocess',preprocessor),('model',LogisticRegression(max_iter=2000,random_state=42))]),
    'Random Forest':Pipeline([('preprocess',preprocessor),('model',RandomForestClassifier(n_estimators=200,class_weight='balanced',random_state=42))])
}
rows=[]
for name,model in models.items():
    model.fit(X_train,y_train); pred=model.predict(X_test)
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,pos_label='Yes'),'Recall':recall_score(y_test,pred,pos_label='Yes'),'F1':f1_score(y_test,pred,pos_label='Yes')})
results=pd.DataFrame(rows).set_index('Model').round(3)
display(results)

## 7. Error analysis

In [ ]:
best_name=results['F1'].idxmax(); best_model=models[best_name]; best_pred=best_model.predict(X_test)
print('Model selected for error inspection:',best_name)
print(classification_report(y_test,best_pred))
ConfusionMatrixDisplay.from_predictions(y_test,best_pred); plt.title(f'Confusion Matrix — {best_name}'); plt.show()
errors=X_test.copy(); errors['actual']=y_test; errors['predicted']=best_pred; errors=errors[errors['actual']!=errors['predicted']]
print('Misclassified test rows:',len(errors)); display(errors)

## 8. Second-year model-selection extension

The separate `model_selection.py` script performs 5-fold stratified cross-validation on the training set and only uses the held-out test set after model selection. This avoids choosing a model based on the test set.

### What I would ship

If Logistic Regression is close to Random Forest under cross-validation, I would ship Logistic Regression because it is simpler and easier to explain on this small dataset. If Random Forest is clearly better, I would choose it and document the trade-off.

### Limitations

This is a small dataset, so performance estimates can vary. It is a baseline, not a production recruitment system. A larger future dataset would justify repeated cross-validation, calibration checks, and monitoring.